<a href="https://colab.research.google.com/github/antoine-jossien/2026_Intro_Python/blob/main/live/part-I/1.6-geospatial-vector-data-practice-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.6-geospatial-vector-data-practice-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# Practice Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Import geopandas as `gpd`, keep track of the crs at every step, and measure only after reprojecting to a metric crs.

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [3]:
# Your solution here

%matplotlib inline
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt


stations_df = pd.DataFrame({
    "name": ["A", "B"],
    "lon": [7.4, 8.5],
    "lat": [46.9, 47.4],
})
stations = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["lon"], stations_df["lat"]),
    crs="EPSG:4326",   # WGS84 longitude/latitude in degrees
)
stations
print("epsg:", stations.crs.to_epsg())
print(stations.geometry)

epsg: 4326
0    POINT (7.4 46.9)
1    POINT (8.5 47.4)
Name: geometry, dtype: geometry


## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [4]:
# Your solution here
from pathlib import Path

Path("_files").mkdir(exist_ok=True)

# écriture
stations.to_file("_files/points.geojson", driver="GeoJSON")

# relecture
reread = gpd.read_file("_files/points.geojson")

print("reread shape:", reread.shape, "| crs preserved:", reread.crs.to_epsg())


reread shape: (2, 4) | crs preserved: 4326


## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [8]:
# Your solution here

%matplotlib inline
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt


# 1. Reprojeter vers EPSG:2056
stations_lv95 = stations.to_crs(2056)

# 2. Afficher le nouveau code EPSG
print("epsg:", stations_lv95.crs.to_epsg())

# 3. Coordonnées projetées du point A (premier point), arrondies au mètre
x = round(stations_lv95.geometry.x.iloc[0])
y = round(stations_lv95.geometry.y.iloc[0])
print("A (m):", x, y)

epsg: 2056
A (m): 2597056 1194322


## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [11]:
# Your solution here

A = stations_lv95.geometry.iloc[0]
B = stations_lv95.geometry.iloc[1]

import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    dist_deg = stations.geometry.distance(stations.geometry.iloc[1])   # Basel..Bern etc., in EPSG:4326

print("A->B distance:", round(A.distance(B)/1000, 1), "km")



A->B distance: 100.2 km


## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [14]:
# Your solution here

from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")

print(stations.crs == poly.crs)

# 2. Jointure spatiale : quels points sont à l'intérieur du polygone ?
inside = gpd.sjoin(stations, poly, predicate="within", how="inner")

# 3. Afficher les noms des points trouvés
print("points inside the polygon:", inside["geometry"].tolist())

True
points inside the polygon: [<POINT (7.4 46.9)>]


## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [15]:
# Your solution here


# 1. Reprojeter vers EPSG:2056
stations_lv95 = stations.to_crs(2056)

# 2. Afficher le nouveau code EPSG
print("epsg:", stations_lv95.crs.to_epsg())

# 3. Coordonnées projetées du point A (premier point), arrondies au mètre
x = round(stations_lv95.geometry.x.iloc[0])
y = round(stations_lv95.geometry.y.iloc[0])
print("A (m):", x, y)


buffers = stations_lv95.buffer(10_000)
print("one buffer:", round(buffers.area.iloc[0] / 1e6), "km^2")


epsg: 2056
A (m): 2597056 1194322
one buffer: 314 km^2


## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
# Your solution here